# CrashLens — CarDD Segmentation: Hard-Class Mask-Quality Tuning

Standalone follow-up to `car_damage_segmentation_cardd_experiments.ipynb`. That notebook's
current run has weak masks on the hard classes (`dent`, `scratch`, `crack`: mask
mAP@0.5:0.95 roughly 0.18-0.29), which caps the severity-fusion extent ratio downstream.

This notebook tries three levers against that and reports a per-class mask mAP table, new
vs current -- **without re-running or depending on the original two experiments**:

1. **Stronger targeted oversampling** -- `copies_per_image = 2` for hard-class training
   images (was 1). Validation and test splits stay untouched.
2. **Tuned Ultralytics built-in augmentation** (mask-aware -- polygons transform
   automatically): mosaic, mixup (~0.1, new), scale, fliplr, hsv, degrees (~10, new).
3. **Optional `yolov8m-seg` candidate** (toggle-controlled) -- a larger backbone, which
   tends to help small/thin instances like scratch and crack.

Kept separate from the original notebook on purpose: that notebook's `run_experiment`
definition lives in the same cell as the original SGD/AdamW training loop, so there is no
way to reach a follow-up section there in a fresh Colab runtime without re-triggering that
training (300 epochs of GPU time) first. This notebook instead:
- reads the dataset the original notebook already converted and saved to Drive
  (`cardd_seg/cardd_yolo_seg_data`) -- no COCO re-conversion;
- reads the current run's comparison table from Drive
  (`cardd_seg/seg_experiment_comparison.csv`) for the baseline -- no in-memory dependency;
- never touches `seg_experiment_comparison.csv`, `combined_comparison_with_previous_reference.csv`,
  `threshold_tuning/`, `final_test/`, or `predictions/` from the current run. Everything
  this notebook writes uses new filenames/run directories.
- does not touch the 31 real expert-priced photos anywhere.

Same protocol as always: fixed `SEED`, `deterministic=True`, select on validation, test
set stays evaluated exactly once (by the original notebook) -- this notebook only compares
on validation. If a candidate here wins, promoting it to a formal test-set evaluation is a
deliberate manual next step, not automatic.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

## 2. Imports, Paths, and Global Settings

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DRIVE_SEG_DIR = Path('/content/drive/MyDrive/cardd_seg')   # same experiment folder
FIGURES_DIR = DRIVE_SEG_DIR / 'figures'                     # same flat figures folder
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CURRENT_RUN_COMPARISON_CSV = DRIVE_SEG_DIR / 'seg_experiment_comparison.csv'
CURRENT_RUN_DATASET_ON_DRIVE = DRIVE_SEG_DIR / 'cardd_yolo_seg_data'  # saved by the original notebook's Section 3

if not CURRENT_RUN_COMPARISON_CSV.exists():
    raise FileNotFoundError(
        f'{CURRENT_RUN_COMPARISON_CSV} not found -- run the original notebook first '
        '(at least through Section 9) so there is a current run to compare against.'
    )
if not CURRENT_RUN_DATASET_ON_DRIVE.exists():
    raise FileNotFoundError(
        f'{CURRENT_RUN_DATASET_ON_DRIVE} not found -- run the original notebook\'s '
        'Section 3 first so the converted dataset exists on Drive.'
    )

# Local working copies (local disk I/O is much faster than reading/writing the Drive
# mount directly during augmentation, same reasoning as the original notebook).
ORIGINAL_YOLO_DIR = Path('/content/cardd_seg_yolo_original')      # local copy of the Drive dataset
TARGET_YOLO_DIR_V2 = Path('/content/cardd_seg_yolo_hardclass_augmented_v2')
BASE_RUN_DIR = Path('/content/cardd_seg_hardclass_tuning_experiments')
BASE_RUN_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'dent',
    'scratch',
    'crack',
    'glass shatter',
    'lamp broken',
    'tire flat'
]
HARD_CLASS_IDS = {0, 1, 2}  # dent, scratch, crack
HARD_CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(HARD_CLASS_IDS)]

# Same training config as the current run, for comparability -- only the levers below change.
SELECTED_IMGSZ = 640
SELECTED_EPOCHS = 150
BATCH_SIZE = 16
PATIENCE = 30

AUGMENTED_COPIES_PER_IMAGE_V2 = 2  # lever 1: was 1 in the current run
TRAIN_LARGER_BACKBONE_CANDIDATE = True  # lever 3: set False to skip the yolov8m-seg candidate

print('Drive segmentation folder:', DRIVE_SEG_DIR)
print('Reading current-run baseline from:', CURRENT_RUN_COMPARISON_CSV)
print('Reading converted dataset from:', CURRENT_RUN_DATASET_ON_DRIVE)

## 3. Copy the Already-Converted Dataset Locally

No COCO re-conversion -- this is the exact dataset `car_damage_segmentation_cardd_experiments.ipynb`'s
Section 3 already built and saved to Drive. Copying it to local `/content` disk once, for
fast I/O during augmentation (same reasoning as the original notebook's local temp dirs).

In [ ]:
if ORIGINAL_YOLO_DIR.exists():
    shutil.rmtree(ORIGINAL_YOLO_DIR)
shutil.copytree(CURRENT_RUN_DATASET_ON_DRIVE, ORIGINAL_YOLO_DIR)
print(f'Copied {CURRENT_RUN_DATASET_ON_DRIVE} -> {ORIGINAL_YOLO_DIR}')

for split in ['train', 'val', 'test']:
    n_images = len(list((ORIGINAL_YOLO_DIR / 'images' / split).glob('*')))
    n_labels = len(list((ORIGINAL_YOLO_DIR / 'labels' / split).glob('*.txt')))
    print(f'{split}: images={n_images}, labels={n_labels}')

## 4. Stronger Hard-Class Oversampling (copies_per_image = 2)

Same augmentation function as the original notebook (photometric-only: brightness/contrast,
mild sharpening, mild Gaussian noise -- polygons stay valid because pixel coordinates are
never touched), just with `copies_per_image = 2` instead of 1. Validation and test splits
are copied through completely unaugmented, same guarantee as before.

In [ ]:
def read_label_classes(label_path: Path):
    classes = set()
    if not label_path.exists():
        return classes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                classes.add(int(parts[0]))
    return classes

def photometric_hardclass_augment(image):
    aug = image.copy()
    alpha = random.uniform(0.90, 1.20)
    beta = random.randint(-12, 18)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)
    if random.random() < 0.7:
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        aug = cv2.filter2D(aug, -1, kernel)
    if random.random() < 0.5:
        noise = np.random.normal(0, 5, aug.shape).astype(np.int16)
        aug = np.clip(aug.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return aug

def build_hardclass_augmented_dataset(original_yolo_dir: Path, target_yolo_dir: Path, hard_class_ids, copies_per_image: int = 1):
    if target_yolo_dir.exists():
        shutil.rmtree(target_yolo_dir)
    shutil.copytree(original_yolo_dir, target_yolo_dir)

    train_images_dir = target_yolo_dir / 'images' / 'train'
    train_labels_dir = target_yolo_dir / 'labels' / 'train'
    image_extensions = {'.jpg', '.jpeg', '.png'}
    train_images = [p for p in train_images_dir.iterdir() if p.suffix.lower() in image_extensions]

    selected_for_augmentation = []
    added_images = 0
    for image_path in train_images:
        label_path = train_labels_dir / image_path.with_suffix('.txt').name
        classes = read_label_classes(label_path)
        if classes & hard_class_ids:
            selected_for_augmentation.append(image_path)
            image = cv2.imread(str(image_path))
            if image is None:
                continue
            for copy_idx in range(1, copies_per_image + 1):
                aug_image = photometric_hardclass_augment(image)
                aug_stem = f'{image_path.stem}_hardaug{copy_idx}'
                cv2.imwrite(str(train_images_dir / f'{aug_stem}{image_path.suffix}'), aug_image)
                shutil.copy(label_path, train_labels_dir / f'{aug_stem}.txt')
                added_images += 1

    return {
        'hard_classes': [CLASS_NAMES[i] for i in sorted(hard_class_ids)],
        'copies_per_image': copies_per_image,
        'selected_train_images': len(selected_for_augmentation),
        'added_augmented_images': added_images
    }

def write_data_yaml(yolo_dir: Path, yaml_path: Path):
    yaml_lines = [
        f'path: {yolo_dir}', 'train: images/train', 'val: images/val', 'test: images/test', '',
        'nc: 6', '', 'names:', '  0: dent', '  1: scratch', '  2: crack',
        '  3: glass shatter', '  4: lamp broken', '  5: tire flat'
    ]
    with open(yaml_path, 'w') as f:
        f.write('\n'.join(yaml_lines) + '\n')
    print(f'Created: {yaml_path}')
    print(open(yaml_path).read())

augmentation_summary_v2 = build_hardclass_augmented_dataset(
    original_yolo_dir=ORIGINAL_YOLO_DIR,
    target_yolo_dir=TARGET_YOLO_DIR_V2,
    hard_class_ids=HARD_CLASS_IDS,
    copies_per_image=AUGMENTED_COPIES_PER_IMAGE_V2
)
print(json.dumps(augmentation_summary_v2, indent=4))

with open(DRIVE_SEG_DIR / 'hardclass_augmentation_summary_v2.json', 'w') as f:
    json.dump(augmentation_summary_v2, f, indent=4)

target_data_yaml_path_v2 = TARGET_YOLO_DIR_V2 / 'data.yaml'
write_data_yaml(TARGET_YOLO_DIR_V2, target_data_yaml_path_v2)

## 5. Leakage Check on the New Dataset

Quick re-verification that augmenting train didn't disturb val/test, same check as the
original notebook's Section 6.

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png'}
train_files = {p.name for p in (TARGET_YOLO_DIR_V2 / 'images/train').iterdir() if p.suffix.lower() in image_extensions}
val_files = {p.name for p in (TARGET_YOLO_DIR_V2 / 'images/val').iterdir() if p.suffix.lower() in image_extensions}
test_files = {p.name for p in (TARGET_YOLO_DIR_V2 / 'images/test').iterdir() if p.suffix.lower() in image_extensions}

overlaps = {
    'train_val_overlap': len(train_files & val_files),
    'train_test_overlap': len(train_files & test_files),
    'val_test_overlap': len(val_files & test_files),
}
print(overlaps)
assert overlaps['train_val_overlap'] == 0 and overlaps['train_test_overlap'] == 0 and overlaps['val_test_overlap'] == 0

with open(DRIVE_SEG_DIR / 'dataset_verification_v2.json', 'w') as f:
    json.dump(overlaps, f, indent=4)

## 6. Tuned Ultralytics Augmentation Hyperparameters

Mosaic and hsv are already near these values by Ultralytics default (`mosaic=1.0,
hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, scale=0.5`) -- passed explicitly so the
config is self-documenting and reproducible rather than relying on library defaults.
`mixup` and `degrees` are the two genuinely new signals (both default to 0.0 / off):

- `mixup=0.10` -- blends two training images+labels; kept mild since CarDD images are
  already visually busy (dents/scratches on reflective paint), so a stronger blend risks
  washing out already-subtle hard-class evidence.
- `degrees=10.0` -- modest rotation jitter; damage orientation isn't semantically fixed
  (a scratch can run any direction), so this is safe here.

In [ ]:
TUNED_AUGMENTATION = {
    'mosaic': 1.0,
    'mixup': 0.10,
    'scale': 0.5,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 10.0,
}
print(json.dumps(TUNED_AUGMENTATION, indent=4))

## 7. Experiment Configuration

Only AdamW is tried (it already beat SGD in the current run), so new-run compute goes
entirely toward the levers actually being tested here, not re-litigating the optimizer.

In [ ]:
BASE_MODEL_S = 'yolov8s-seg.pt'  # same backbone size as the current run, for comparability

EXPERIMENTS_V2 = [
    {
        'phase': 'cardd_seg_hardclass_mask_quality_v2',
        'name': f'yolov8s_seg_img{SELECTED_IMGSZ}_hardaug2x_tunedaug_adamw_ep{SELECTED_EPOCHS}',
        'dataset_variant': 'hardclass_augmented_2x_tuned_aug',
        'model': BASE_MODEL_S,
        'imgsz': SELECTED_IMGSZ,
        'epochs': SELECTED_EPOCHS,
        'batch': BATCH_SIZE,
        'patience': PATIENCE,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'weight_decay': 0.01,
        'extra_train_kwargs': TUNED_AUGMENTATION,
    },
]

if TRAIN_LARGER_BACKBONE_CANDIDATE:
    EXPERIMENTS_V2.append({
        'phase': 'cardd_seg_hardclass_mask_quality_v2',
        'name': f'yolov8m_seg_img{SELECTED_IMGSZ}_hardaug2x_tunedaug_adamw_ep{SELECTED_EPOCHS}',
        'dataset_variant': 'hardclass_augmented_2x_tuned_aug',
        'model': 'yolov8m-seg.pt',
        'imgsz': SELECTED_IMGSZ,
        'epochs': SELECTED_EPOCHS,
        'batch': BATCH_SIZE,
        'patience': PATIENCE,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'weight_decay': 0.01,
        'extra_train_kwargs': TUNED_AUGMENTATION,
    })

with open(DRIVE_SEG_DIR / 'experiments_config_v2.json', 'w') as f:
    json.dump(EXPERIMENTS_V2, f, indent=4)

print(json.dumps(EXPERIMENTS_V2, indent=4))

## 8. Training and Validation Loop

In [ ]:
from ultralytics import YOLO

FIGURE_FILENAME_PATTERNS = [
    'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'labels.jpg', 'labels_correlogram.jpg',
    'BoxP_curve.png', 'BoxR_curve.png', 'BoxPR_curve.png', 'BoxF1_curve.png',
    'MaskP_curve.png', 'MaskR_curve.png', 'MaskPR_curve.png', 'MaskF1_curve.png',
    'val_batch*_labels.jpg', 'val_batch*_pred.jpg'
]

def copy_run_figures_to_drive(run_source_dir: Path, figures_dir: Path, run_label: str):
    figures_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for pattern in FIGURE_FILENAME_PATTERNS:
        for src_path in run_source_dir.rglob(pattern):
            dst_name = f'{run_label}__{src_path.name}'
            shutil.copy(src_path, figures_dir / dst_name)
            copied.append(dst_name)
    return copied


def run_experiment(exp, data_yaml_path: Path):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name = f"{exp['name']}_{timestamp}"
    local_run = BASE_RUN_DIR / run_name
    drive_run = DRIVE_SEG_DIR / run_name
    local_run.mkdir(parents=True, exist_ok=True)

    with open(local_run / 'experiment_config.json', 'w') as f:
        json.dump(exp, f, indent=4)

    print(f"\nStarting experiment: {run_name}")
    model = YOLO(exp['model'])

    train_kwargs = {
        'data': str(data_yaml_path),
        'imgsz': exp['imgsz'],
        'epochs': exp['epochs'],
        'batch': exp['batch'],
        'patience': exp['patience'],
        'project': str(local_run / 'training'),
        'name': 'train',
        'exist_ok': True,
        'seed': SEED,
        'deterministic': True,
        'optimizer': exp['optimizer'],
        'lr0': exp['lr0'],
        'weight_decay': exp['weight_decay']
    }
    if 'momentum' in exp:
        train_kwargs['momentum'] = exp['momentum']
    train_kwargs.update(exp.get('extra_train_kwargs', {}))

    model.train(**train_kwargs)

    best_weights = local_run / 'training' / 'train' / 'weights' / 'best.pt'
    if not best_weights.exists():
        raise FileNotFoundError(f'Best weights not found: {best_weights}')

    model = YOLO(str(best_weights))
    val_result = model.val(
        data=str(data_yaml_path), split='val', imgsz=exp['imgsz'],
        project=str(local_run / 'validation'), name='val', exist_ok=True
    )

    summary = {
        'run_name': run_name,
        'source': 'cardd_segmentation_hardclass_tuning_v2',
        'weights_path': str(best_weights),
        **exp,
        'val_box_mAP50': float(val_result.box.map50),
        'val_box_mAP50_95': float(val_result.box.map),
        'val_box_precision': float(val_result.box.mp),
        'val_box_recall': float(val_result.box.mr),
        'val_seg_mAP50': float(val_result.seg.map50),
        'val_seg_mAP50_95': float(val_result.seg.map),
        'val_seg_precision': float(val_result.seg.mp),
        'val_seg_recall': float(val_result.seg.mr)
    }
    for class_name, value in zip(CLASS_NAMES, val_result.box.maps):
        summary[f'val_box_map50_95_{class_name}'] = float(value)
    for class_name, value in zip(CLASS_NAMES, val_result.seg.maps):
        summary[f'val_seg_map50_95_{class_name}'] = float(value)

    with open(local_run / 'summary.json', 'w') as f:
        json.dump(summary, f, indent=4)

    if drive_run.exists():
        shutil.rmtree(drive_run)
    shutil.copytree(local_run, drive_run)
    summary['weights_path'] = str(drive_run / 'training' / 'train' / 'weights' / 'best.pt')

    copied_figures = copy_run_figures_to_drive(drive_run, FIGURES_DIR, run_name)
    print(f'Copied {len(copied_figures)} figure(s) to {FIGURES_DIR}')
    print(f'Saved to Drive: {drive_run}')
    return summary

all_results_v2 = []
for exp in EXPERIMENTS_V2:
    summary = run_experiment(exp, target_data_yaml_path_v2)
    all_results_v2.append(summary)

new_results_v2_df = pd.DataFrame(all_results_v2).sort_values(
    by='val_seg_mAP50_95', ascending=False
).reset_index(drop=True)

# New filename -- never touches seg_experiment_comparison.csv from the current run.
new_results_v2_df.to_csv(DRIVE_SEG_DIR / 'seg_experiment_comparison_v2_hardclass_tuning.csv', index=False)
display(new_results_v2_df)

## 9. Per-Class Mask mAP@0.5:0.95: New vs Current

The current run's comparison table is read straight from Drive, not from in-memory
state -- this is the only coupling to the original notebook, and it's read-only.

In [ ]:
current_comparison_df = pd.read_csv(CURRENT_RUN_COMPARISON_CSV)
current_best = current_comparison_df.sort_values(by='val_seg_mAP50_95', ascending=False).iloc[0]

def per_class_mask_map(row):
    return {
        class_name: row[f'val_seg_map50_95_{class_name}']
        for class_name in CLASS_NAMES
        if f'val_seg_map50_95_{class_name}' in row
    }

comparison_columns = {'current_run (' + str(current_best['run_name']) + ')': per_class_mask_map(current_best)}
for _, row in new_results_v2_df.iterrows():
    comparison_columns[row['run_name']] = per_class_mask_map(row)

per_class_comparison_df = pd.DataFrame(comparison_columns)
per_class_comparison_df.index.name = 'class'

current_col = per_class_comparison_df.columns[0]
for new_col in per_class_comparison_df.columns[1:]:
    per_class_comparison_df[f'delta_{new_col}'] = per_class_comparison_df[new_col] - per_class_comparison_df[current_col]

display(per_class_comparison_df)

hardclass_rows = per_class_comparison_df.loc[per_class_comparison_df.index.isin(HARD_CLASS_NAMES)]
print('\nHard classes only (dent/scratch/crack):')
display(hardclass_rows)

per_class_comparison_df.to_csv(DRIVE_SEG_DIR / 'hardclass_tuning_per_class_mask_map_comparison.csv')

In [ ]:
per_class_comparison_df[[c for c in per_class_comparison_df.columns if not c.startswith('delta_')]].plot(
    kind='bar', figsize=(10, 5)
)
plt.title('Per-Class Mask mAP@0.5:0.95 — Current Run vs Hard-Class Tuning Candidates')
plt.ylabel('Mask mAP@0.5:0.95')
plt.xlabel('Damage Class')
plt.xticks(rotation=45)
plt.legend(title='run', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(DRIVE_SEG_DIR / 'hardclass_tuning_per_class_mask_map_comparison.png', dpi=200)
plt.savefig(FIGURES_DIR / 'hardclass_tuning_per_class_mask_map_comparison.png', dpi=200)
plt.show()

## 10. Final Output Structure

New files only -- nothing from the current run is overwritten:

```text
cardd_seg/
├── hardclass_augmentation_summary_v2.json
├── dataset_verification_v2.json
├── experiments_config_v2.json
├── seg_experiment_comparison_v2_hardclass_tuning.csv
├── hardclass_tuning_per_class_mask_map_comparison.csv
├── hardclass_tuning_per_class_mask_map_comparison.png
├── figures/                 (new run figures appended, prefixed by run name)
└── <each new training run>/
    ├── experiment_config.json
    ├── summary.json
    ├── training/
    └── validation/
```